# 05 — Customer / Inventory Segmentation (Clustering)**Step 8 of the brief.** Unsupervised: no target, no answer key. We're looking for thenatural groupings a buyer actually shops along.A note on the brief's wording: it says *customer* segmentation, but the dataset has nocustomers — only diamonds. What we can genuinely do is segment the **inventory intoproduct tiers**, which is the thing a merchandiser would actually build price bands andmarketing around. Say that plainly in your README rather than pretending otherwise.

In [1]:
import sys, pathlibsys.path.append(str(pathlib.Path.cwd().parent))   # [so `from src...` works from notebooks/]import numpy as np, pandas as pdimport matplotlib.pyplot as plt, seaborn as snsfrom src.config import *sns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (9, 5)pd.set_option("display.max_columns", 40)

SyntaxError: invalid syntax (2965902560.py, line 1)

In [2]:
from sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeans, DBSCANfrom sklearn.decomposition import PCAfrom sklearn.metrics import silhouette_scorefrom src.data_prep import build, CLU_FEATURESdf = build(verbose=False)X = StandardScaler().fit_transform(df[CLU_FEATURES])print("Segmenting on:", CLU_FEATURES)print("Matrix:", X.shape)

SyntaxError: invalid syntax (566851883.py, line 1)

**Scaling is mandatory and non-negotiable here.** `price` runs to 18,823 and `cut_rank` runs0–4. Euclidean distance squares the difference in every dimension and sums them. Unscaled,price would be the *only* feature that mattered and every other column would be noise. Youwould produce a clustering of price, dressed up as a clustering of diamonds — and nothing woulderror out to tell you.

## Elbow + silhouette

In [ ]:
ks, inertias, sils = range(2, 11), [], []for k in ks:    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)    inertias.append(km.inertia_)    sils.append(silhouette_score(X, km.labels_, sample_size=10000, random_state=RANDOM_STATE))fig, ax = plt.subplots(1, 2, figsize=(14, 4))ax[0].plot(ks, inertias, "o-", color=PALETTE["primary"])ax[0].set_title("Elbow (inertia)"); ax[0].set_xlabel("k")ax[1].plot(ks, sils, "o-", color=PALETTE["secondary"])ax[1].set_title("Silhouette"); ax[1].set_xlabel("k")plt.tight_layout(); plt.show()for k, i, s in zip(ks, inertias, sils):    print(f"k={k}  inertia={i:12,.0f}  silhouette={s:.4f}")

**Inertia always falls as k rises** — at k = n_rows every point is its own cluster and inertia isexactly 0. So you cannot minimise it; the minimum is a useless answer. The **elbow** is whereextra clusters stop buying real structure and start merely subdividing coherent groups.Elbows are notoriously ambiguous to eyeball, which is why silhouette is plotted beside it.Silhouette asks each point: *how close am I to my own cluster, versus the nearest other one?*It runs −1 to +1 and has a genuine maximum. **Where the bend and the peak agree, you have adefensible k.** Where they disagree, say so and pick on business grounds.

## Fit and profile

In [ ]:
K = 4   # [set from the plots above]km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)df["segment"] = km.labels_profile = df.groupby("segment").agg(    n=("price", "size"),    carat=("carat", "mean"),    price=("price", "mean"),    price_med=("price", "median"),    cut=("cut_rank", "mean"),    color=("color_rank", "mean"),    clarity=("clarity_rank", "mean"),).round(2)profile["share"] = (profile["n"] / len(df) * 100).round(1)display(profile)

**A cluster label is meaningless until you describe it.** `segment == 2` tells a merchandisernothing. The profile table is where clustering stops being a maths exercise and becomes abusiness artefact. Read the rows and *name* them:- small carat, low price, high grades → **"Entry engagement"** — the volume tier- large carat, high price, mid grades → **"Statement stones"** — big, and flawed *because* big- mid carat, high grades → **"Quality-first buyer"** — pays for the certificate, not the size- mid carat, low grades, low price → **"Value seeker"** — size for the moneyNote how the *carat–clarity confound from the EDA reappears here on its own*: the big-stonecluster carries the lower average clarity. K-Means found the same real-world structure thatthe correlation matrix hinted at, without being told to look for it.

## Visualise

In [ ]:
pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))s = np.random.RandomState(RANDOM_STATE).choice(len(X), 8000, replace=False)ax[0].scatter(pcs[s, 0], pcs[s, 1], c=km.labels_[s], cmap="cool", s=8, alpha=.6)ax[0].set_title(f"Segments in PCA space (k={K})"); ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2")for seg in range(K):    m = df["segment"] == seg    ax[1].scatter(df.loc[m, "carat"], df.loc[m, "price"], s=5, alpha=.35, label=f"seg {seg}")ax[1].set_xlabel("carat"); ax[1].set_ylabel("price $")ax[1].set_title("Segments on the two axes a buyer actually uses"); ax[1].legend()plt.tight_layout(); plt.show()

**PCA is used only for the picture, never for the fitting.** We clustered in the full 6-dimensionalspace and are viewing its least-lossy 2D shadow. Reducing *first* and clustering second wouldthrow away information the clusters might need.

## DBSCAN — for contrast, and to show why K-Means is the right call here

In [ ]:
from sklearn.neighbors import NearestNeighborssub = X[np.random.RandomState(RANDOM_STATE).choice(len(X), 10000, replace=False)]nn = NearestNeighbors(n_neighbors=12).fit(sub)d, _ = nn.kneighbors(sub)plt.plot(np.sort(d[:, -1]), color=PALETTE["primary"])plt.title("k-distance plot — the bend is your eps"); plt.ylabel("distance to 12th neighbour"); plt.show()for eps in [0.3, 0.5, 0.7, 1.0, 1.5]:    db = DBSCAN(eps=eps, min_samples=12).fit(sub)    lab = db.labels_    n = len(set(lab)) - (1 if -1 in lab else 0)    noise = (lab == -1).sum()    sil = silhouette_score(sub, lab) if n > 1 else float("nan")    print(f"eps={eps:4.1f}  clusters={n:3d}  noise={noise:5d}/{len(sub)}  silhouette={sil:.4f}")

**DBSCAN's honest verdict on this dataset: it is the wrong tool, and knowing why is the point.**DBSCAN assumes clusters are **dense regions separated by sparse ones**. But the diamonds data isone **continuous cloud** — price and carat vary smoothly, with no empty gaps between the tiers.There are no natural density valleys for DBSCAN to find, so it either lumps everything into onecluster or declares half the inventory noise, depending on `eps`.K-Means, by contrast, **doesn't require gaps.** It happily partitions a continuous cloud into kcompact regions, which is exactly what a merchandiser wants: price bands are *imposed* on acontinuum, not discovered in it.**The lesson generalises: match the algorithm to the geometry.** DBSCAN wins when clusters areirregularly shaped, when k is unknown, and when outliers must be *identified* rather thanabsorbed. K-Means wins on roughly spherical, comparably-dense, continuous data where you want afixed number of segments. Diamonds are the second case. Being able to state that — and show theevidence — is worth more than a clean run.

In [ ]:
df[["carat", "cut", "color", "clarity", "price", "segment"]].to_csv(    RESULT_DIR / "segments.csv", index=False)profile.to_csv(RESULT_DIR / "segment_profiles.csv")pd.Series({"k": K, "silhouette": float(sils[K - 2])}).to_json(RESULT_DIR / "clustering_results.json")print("saved segments + profiles")